# 00 — Protocol and governance gate

This notebook freezes the computational context and reports the submission
governance gate before any result is interpreted. The primary estimand is
post-development, within-GAVD cross-validated performance on held-out
**source videos**. A source video is
the independent unit; persistent person identifiers are unavailable, so the
workflow cannot support an unseen-person claim. Folder names are dataset
annotations, not diagnoses, and no diagnostic or clinical claim is made.

The protocol separates three claims that must not be collapsed: an analytic
wrapper can force odd output; an anatomy-aware encoder-plus-probe can exhibit
useful native odd behavior; and a checkpoint can satisfy a direct strict
token-equivariance test. Only the last two are empirical, and neither removes
the BlazePose schema, preprocessing, architecture, or source-sampling
assumptions supplied to the experiment.

The checked-in governance record intentionally fails closed until an
institutional ethics determination, a data-use review, and a derived-pose
release review each have a dated internal reference. Public availability is
not a substitute for those determinations. This suite never redistributes
raw video or identity-bearing frames. Linkable identifiers, derived poses,
embeddings, and checkpoints may be released only as completed reviews permit.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
from IPython import get_ipython
from IPython.display import display


def locate_suite_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for ancestor in (start, *start.parents):
        for candidate in (ancestor, ancestor / "neurips-laterality"):
            if (
                (candidate / "config" / "protocol.json").is_file()
                and (candidate / "laterality").is_dir()
            ):
                return candidate.resolve()
    raise FileNotFoundError(
        "Could not locate neurips-laterality from the current working directory."
    )


SUITE_ROOT = locate_suite_root()
if str(SUITE_ROOT) not in sys.path:
    sys.path.insert(0, str(SUITE_ROOT))

from laterality.config import load_context

context = load_context(SUITE_ROOT / "config" / "protocol.json")
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")


def show_inline(figure):
    display(figure)
    plt.close(figure)


print(
    f"suite={SUITE_ROOT} profile={context.profile} "
    f"artifacts={context.artifact_root} protocol={context.protocol_digest[:12]}"
)

In [ ]:
from laterality.config import model_config
from laterality.governance import load_governance, submission_readiness
from laterality.visualization import governance_figure

governance_path = SUITE_ROOT / "governance" / "status.json"
governance_payload = load_governance(governance_path)
governance_readiness = submission_readiness(governance_payload)

protocol_snapshot = {
    "profile": context.profile,
    "synthetic_smoke": not context.is_paper,
    "protocol_digest": context.protocol_digest,
    "claim": context.protocol["claim_boundary"]["primary"],
    "unsupported_claims": context.protocol["claim_boundary"]["not_supported"],
    "independent_unit": context.protocol["claim_boundary"]["independent_unit"],
    "selected_folds": list(context.folds),
    "selected_seeds": list(context.seeds),
    "selected_variants": list(context.variants),
    "model": model_config(context),
    "primary_lane": context.protocol["evaluation"]["primary_lane"],
    "constructed_repair_lane": context.protocol["evaluation"][
        "constructed_repair_lane"
    ],
    "primary_seed_estimand": context.protocol["evaluation"][
        "primary_seed_estimand"
    ],
    "representation_equivariance": context.protocol["evaluation"][
        "representation_equivariance"
    ],
    "decision_rules": context.protocol["evaluation"]["decision_rules"],
}
show_inline(governance_figure(context, governance_payload))
protocol_snapshot, governance_readiness

A `smoke` run uses generated poses and a tiny model only to test plumbing,
lineage checks, and algebraic invariants. Its metrics are **not empirical
evidence** and must never enter a paper table. Likewise, a completed paper
run does not override an unresolved governance gate.